To make this model work, we need to do the following: 
* For all crashes in our study area (Oakland + Berkeley), we need to mark whether a crash is associated with a traffic island and signal
* We need to determine which other explanatory variables to include in the model:
    * Traffic signal presence (OSM)
    * Traffic refuge island presence (OSM)
    * Number of lanes (TBD)
        * Looks like you have to play with the overpass API to get this data from OSM https://stackoverflow.com/questions/56558717/query-all-roads-with-overpass-api-and-export-as-polygon
    * Ped characteristics (gender, age, etc.) (retrieve from SWITRS)
        * **How do we do this if ped characteristics are tied to crash party but we want to look at crashes overall?**
    * AADT (From replica)
    * Functional classification of road as a proxy for speed
    * Day of the week (SWITRS)
    * Lighting at intersection (TBD, possibly on OSM)

In [95]:
import pandas as pd
import numpy as np
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

In [96]:
# Load dataset
file_path = "crash_data_for_model.csv"  # Update with the actual file path
df = pd.read_csv(file_path, dtype=str)

In [97]:
# Remove irrelevant columns

df = df[[
        'COLLISION_SEVERITY',
    'AT_FAULT', # No should be reference
    'PARTY_SEX', # Male should be reference
    'PARTY_AGE',
    'RACE', # White should be reference
    'LIGHTING', # Daylight should be reference
    'DAY_OF_WEEK',
    'WEATHER_1', # Clear should be reference
    'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    'PARTY_NUMBER_KILLED',
    'PARTY_NUMBER_INJURED',
    'road_class',
    'island_id', # Convert to 1/0
    'volume', 
    'signal_id' # Convert to 1/0
]]

In [98]:
def remap_severity(severity_series):
    # Convert severity values to numeric, coercing errors to NaN
    severity_numeric = pd.to_numeric(severity_series, errors="coerce")
    # Define the mapping from original to reversed scale
    severity_mapping = {1: 4, 2: 3, 3: 2, 4: 1}
    remapped = severity_numeric.map(severity_mapping)
    # Define the order for the categorical variable
    severity_order = [1, 2, 3, 4]
    return pd.Categorical(remapped, categories=severity_order, ordered=True)

# Apply severity remapping using the dedicated function
df["COLLISION_SEVERITY"] = remap_severity(df["COLLISION_SEVERITY"])

In [99]:
# # Collapse DAY_OF_WEEK into "Weekday" (Mon-Fri) vs. "Weekend" (Sat-Sun)
# df["DAY_OF_WEEK"] = df["DAY_OF_WEEK"].apply(lambda x: "Weekday" if x in ["1", "2", "3", "4", "5"] else "Weekend")

In [100]:
# Remap at fault category
fault_dict = {'N':0, 'Y':1}
df['AT_FAULT'] = df['AT_FAULT'].map(fault_dict)
df['AT_FAULT'] = df['AT_FAULT']#.astype(int)
df['AT_FAULT'].value_counts()

AT_FAULT
0    610
1     60
Name: count, dtype: int64

In [101]:
# Remap island data
print(df['island_id'].value_counts().sum())
df['island_id'] = df['island_id'].notna()#.astype(int)
df['island_id'] = df['island_id'].map(int)
print(df['island_id'].value_counts())


36
island_id
0    634
1     36
Name: count, dtype: int64


In [102]:
# Remap signal data
print(df['signal_id'].value_counts().sum())
df['signal_id'] = df['signal_id'].notna()#.astype(int)
df['signal_id'] = df['signal_id'].map(int)
print(df['signal_id'].value_counts())

169
signal_id
0    501
1    169
Name: count, dtype: int64


In [103]:
# Remap ped_action data
print(df['PED_ACTION'].value_counts())
df.loc[df['PED_ACTION']!='B', 'PED_ACTION'] = 0
df.loc[df['PED_ACTION']=='B', 'PED_ACTION'] = 1 # B - Crossing in Crosswalk at Intersection
df['PED_ACTION'].value_counts()

PED_ACTION
B    511
E     53
F     37
D     35
C     27
-      6
G      1
Name: count, dtype: int64


PED_ACTION
1    511
0    159
Name: count, dtype: int64

In [104]:
# Remap ped_action data
print(df['PRIMARY_COLL_FACTOR'].value_counts())
df.loc[df['PRIMARY_COLL_FACTOR']!='A', 'PRIMARY_COLL_FACTOR'] = 0
df.loc[df['PRIMARY_COLL_FACTOR']=='A', 'PRIMARY_COLL_FACTOR'] = 1 # A - (Vehicle) Code Violation
df['PRIMARY_COLL_FACTOR'].value_counts()

PRIMARY_COLL_FACTOR
A    639
D     26
B      2
C      2
-      1
Name: count, dtype: int64


PRIMARY_COLL_FACTOR
1    639
0     31
Name: count, dtype: int64

In [105]:
# Remap ped_action data
print(df['WEATHER_1'].value_counts())
df.loc[df['WEATHER_1']!='A', 'WEATHER_1'] = 0
df.loc[df['WEATHER_1']=='A', 'WEATHER_1'] = 1 # A - Clear
df['WEATHER_1'].value_counts()

WEATHER_1
A    558
B     56
C     41
-     13
F      1
E      1
Name: count, dtype: int64


WEATHER_1
1    558
0    112
Name: count, dtype: int64

In [106]:
# Remap ped_action data
print(df['LIGHTING'].value_counts())
df.loc[df['LIGHTING']!='A', 'LIGHTING'] = 0
df.loc[df['LIGHTING']=='A', 'LIGHTING'] = 1 # A - Daylight
df['LIGHTING'].value_counts()

LIGHTING
A    421
C    195
B     30
D     12
E      6
-      6
Name: count, dtype: int64


LIGHTING
1    421
0    249
Name: count, dtype: int64

In [107]:
# Apply dummy encoding
df = pd.get_dummies(df, columns=[
    'PARTY_SEX', # Male should be reference
    'RACE', # White should be reference
    # 'LIGHTING', # Daylight should be reference
    # 'WEATHER_1', # Clear should be reference
    # 'PRIMARY_COLL_FACTOR', # Make dummy for A - Vehicle Code Violation
    # 'PED_ACTION', # B - Crossing in Crosswalk at Intersection should be reference
    ], drop_first=True)

display(df)

,COLLISION_SEVERITY,AT_FAULT,PARTY_AGE,LIGHTING,DAY_OF_WEEK,WEATHER_1,PRIMARY_COLL_FACTOR,PED_ACTION,PARTY_NUMBER_KILLED,PARTY_NUMBER_INJURED,...,island_id,volume,signal_id,PARTY_SEX_F,PARTY_SEX_M,PARTY_SEX_X,RACE_B,RACE_H,RACE_O,RACE_W
0,1,0,60,1,2,0,1,1,0,1,...,0,188606.62996098422,0,False,True,False,False,False,False,True
1,1,0,57,1,1,1,1,1,0,1,...,0,1183902.7845568222,0,False,True,False,True,False,False,False
2,2,0,30,0,5,1,1,1,0,1,...,0,1183902.7845568222,0,True,False,False,False,True,False,False
3,2,1,13,1,7,1,1,1,0,1,...,0,1183902.7845568222,0,True,False,False,False,True,False,False
4,1,0,23,1,3,1,1,0,0,1,...,0,3262.780625864375,0,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,2,0,45,1,4,1,1,1,0,1,...,0,605870.8697948474,0,True,False,False,False,False,False,True
666,3,0,77,1,2,1,1,1,0,1,...,0,33858.13232104739,0,True,False,False,False,False,False,True
667,2,0,42,1,6,1,1,1,0,1,...,0,266336.5801759236,0,True,False,False,False,False,False,True
668,1,0,38,0,4,0,1,1,0,1,...,0,560856.5575155583,0,False,True,False,False,False,False,True


In [112]:
cols = ['PARTY_AGE', 
        'DAY_OF_WEEK', 
        'PARTY_NUMBER_KILLED', 
        'PARTY_NUMBER_INJURED', 
        'road_class']

df[cols] = df[cols].astype(int)

df['volume'] = df['volume'].astype(float)
df['volume'] = df['volume'].fillna(0).astype(int)
# lets scale the volumes between 0 and 1 so they aren't such ridiculous numeric outliers
max_value = df['volume'].max()
df['volume'] = df['volume']/max_value

# let's convert all the True/False into 1/0
for col in df:
    if df[col].dtype == bool:
        df[col] = df[col].map(int)
for col in df:
    if df[col].dtype == object:
        df[col] = df[col].map(int)

df.dtypes

COLLISION_SEVERITY      category
AT_FAULT                   int64
PARTY_AGE                  int64
LIGHTING                   int64
DAY_OF_WEEK                int64
WEATHER_1                  int64
PRIMARY_COLL_FACTOR        int64
PED_ACTION                 int64
PARTY_NUMBER_KILLED        int64
PARTY_NUMBER_INJURED       int64
road_class                 int64
island_id                  int64
volume                   float64
signal_id                  int64
PARTY_SEX_F                int64
PARTY_SEX_M                int64
PARTY_SEX_X                int64
RACE_B                     int64
RACE_H                     int64
RACE_O                     int64
RACE_W                     int64
dtype: object

In [115]:
# Define independent variables (all columns except the target)
independent_vars = df.columns.difference([
    'COLLISION_SEVERITY',
    'PARTY_NUMBER_INJURED','PARTY_NUMBER_KILLED', # These are too similar to collision severity
    ])

independent_vars

Index(['AT_FAULT', 'DAY_OF_WEEK', 'LIGHTING', 'PARTY_AGE', 'PARTY_SEX_F',
       'PARTY_SEX_M', 'PARTY_SEX_X', 'PED_ACTION', 'PRIMARY_COLL_FACTOR',
       'RACE_B', 'RACE_H', 'RACE_O', 'RACE_W', 'WEATHER_1', 'island_id',
       'road_class', 'signal_id', 'volume'],
      dtype='object')

In [116]:
# Specify and calibrate the Ordered Logit Model
model = OrderedModel(
    endog=df["COLLISION_SEVERITY"],
    exog=df[independent_vars],
    distr="logit"
)

# I tried some different model methods. I think we may have too many un-useful independent variables
result = model.fit(method="cg")
# result = model.fit(method="nm")
# result = model.fit(method="bfgs")
print(result.summary())

/opt/anaconda3/lib/python3.12/site-packages/scipy/optimize/_optimize.py:1659: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_cg(f, x0, args, fprime, callback=callback, c1=c1, c2=c2,


         Current function value: 1.033535
         Iterations: 500
         Function evaluations: 1164
         Gradient evaluations: 1155
                             OrderedModel Results                             
Dep. Variable:     COLLISION_SEVERITY   Log-Likelihood:                -692.47
Model:                   OrderedModel   AIC:                             1427.
Method:            Maximum Likelihood   BIC:                             1522.
Date:                Mon, 20 Apr 2026                                         
Time:                        13:26:15                                         
No. Observations:                 670                                         
Df Residuals:                     649                                         
Df Model:                          18                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [15]:
# Compute and display Odds Ratios
odds_ratios = np.exp(result.params)
print("\nOdds Ratios:\n", odds_ratios)

# Predict probabilities for each severity level
predicted_probs = result.predict()

# Convert predictions to a DataFrame
predicted_probs_df = pd.DataFrame(predicted_probs)

# Determine the most likely severity level for each observation
df["Predicted_Severity"] = predicted_probs_df.idxmax(axis=1)

# Display performance metrics with a confusion matrix (objective results only)
confusion_matrix = pd.crosstab(df["COLLISION_SEVERITY"], df["Predicted_Severity"],
                               rownames=['Actual'], colnames=['Predicted'])
print("\nConfusion Matrix:\n", confusion_matrix)



Odds Ratios:
 AT_FAULT                   1.191208
DAY_OF_WEEK                0.940323
LIGHTING_A                 0.620546
LIGHTING_B                 0.542245
LIGHTING_C                 0.678508
LIGHTING_D                 1.403706
LIGHTING_E                 3.937060
PARTY_AGE                  1.000646
PARTY_NUMBER_INJURED       0.121309
PARTY_NUMBER_KILLED      188.016001
PARTY_SEX_F                1.387540
PARTY_SEX_M                1.430867
PARTY_SEX_X                0.655856
PED_ACTION_B               2.102877
PED_ACTION_C               0.494435
PED_ACTION_D               3.893584
PED_ACTION_E               2.194725
PED_ACTION_F               1.351453
PED_ACTION_G               0.780577
PRIMARY_COLL_FACTOR_A      1.020327
PRIMARY_COLL_FACTOR_B      1.975209
PRIMARY_COLL_FACTOR_C      2.394715
PRIMARY_COLL_FACTOR_D      0.787599
RACE_B                     0.645907
RACE_H                     0.746349
RACE_O                     1.564526
RACE_W                     1.588035
WEATHER_1_A  

In [16]:
display(df)

,COLLISION_SEVERITY,AT_FAULT,PARTY_AGE,DAY_OF_WEEK,PARTY_NUMBER_KILLED,PARTY_NUMBER_INJURED,road_class,island_id,volume,signal_id,...,PRIMARY_COLL_FACTOR_B,PRIMARY_COLL_FACTOR_C,PRIMARY_COLL_FACTOR_D,PED_ACTION_B,PED_ACTION_C,PED_ACTION_D,PED_ACTION_E,PED_ACTION_F,PED_ACTION_G,Predicted_Severity
0,1,0,60,2,0,1,4,0,0.003383,0,...,0,0,0,1,0,0,0,0,0,1
1,1,0,57,1,0,1,5,0,0.021233,0,...,0,0,0,1,0,0,0,0,0,0
2,2,0,30,5,0,1,5,0,0.021233,0,...,0,0,0,1,0,0,0,0,0,0
3,2,1,13,7,0,1,5,0,0.021233,0,...,0,0,0,1,0,0,0,0,0,0
4,1,0,23,3,0,1,7,0,0.000059,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,2,0,45,4,0,1,4,0,0.010866,0,...,0,0,0,1,0,0,0,0,0,1
666,3,0,77,2,0,1,5,0,0.000607,0,...,0,0,0,1,0,0,0,0,0,1
667,2,0,42,6,0,1,4,0,0.004777,0,...,0,0,0,1,0,0,0,0,0,1
668,1,0,38,4,0,1,4,0,0.010059,0,...,0,0,0,1,0,0,0,0,0,1
